# Course 3 — Recommender Systems
=================================

Recommender systems suggest items to users based on their preferences. There are two primary paradigms: content-based filtering (utilizing descriptive features of items and users) and collaborative filtering (utilizing patterns of user interactions).

In [8]:
import sys
from pathlib import Path
# Add repository root to sys.path dynamically
project_root = Path(".").resolve()
while project_root.name and not (project_root / "utils").is_dir():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity


## 1. Content-Based Filtering

In content-based filtering, we assume we have descriptive feature vectors $\mathbf{x}^{(i)}$ for each item $i$ (e.g., romance and action scores for movies). We learn a preference vector $\mathbf{w}^{(j)}$ and bias $b^{(j)}$ for each user $j$.

The predicted rating of user $j$ for item $i$ is calculated as a linear model:
$$\hat{y}^{(i,j)} = \mathbf{w}^{(j)} \cdot \mathbf{x}^{(i)} + b^{(j)}$$

In [9]:
# ── 1. Content-Based Filtering ────────────────────────────────────────
# Predict rating = w·x + b  where x = features of item, w = user preferences

print("── Content-Based Filtering ──")

# Items: [romance, action]  (2 features)
items = np.array([
    [0.9, 0.1],   # Movie A: very romantic
    [0.1, 0.9],   # Movie B: very action
    [0.5, 0.5],   # Movie C: balanced
    [0.8, 0.3],   # Movie D: mostly romantic
    [0.2, 0.7],   # Movie E: mostly action
])
movie_names = ["Movie A", "Movie B", "Movie C", "Movie D", "Movie E"]

# User preferences (w)
user_w = np.array([0.8, 0.3])   # likes romance more than action
user_b = 0.0

ratings = items @ user_w + user_b
print("Predicted ratings:")
for name, rating in zip(movie_names, ratings):
    print(f"  {name}: {rating:.2f}")

── Content-Based Filtering ──
Predicted ratings:
  Movie A: 0.75
  Movie B: 0.35
  Movie C: 0.55
  Movie D: 0.73
  Movie E: 0.37


## 2. Collaborative Filtering & Matrix Factorization

In collaborative filtering, we do not require predefined features for items or users. Instead, we learn a set of latent features $\mathbf{x}^{(i)} \in \mathbb{R}^d$ and user preference weights $\mathbf{w}^{(j)} \in \mathbb{R}^d$ simultaneously from a sparse ratings matrix $R$.

This is called **Matrix Factorization** because we approximate the ratings matrix as the product of two lower-rank matrices: $R \approx U V^T$, where $U$ represents the user factors and $V$ represents the item factors.

### Optimization Objective
We minimize the reconstruction error over all observed ratings (where $r(i,j)=1$, indicating user $j$ rated item $i$) along with L2 regularization to prevent overfitting:
$$J(U, V) = \frac{1}{2} \sum_{(i,j): r(i,j)=1} \left( \mathbf{u}^{(j)} \cdot \mathbf{v}^{(i)} - R_{i,j} \right)^2 + \frac{\lambda}{2} \sum_{j=1}^{n_u} ||\mathbf{u}^{(j)}||^2 + \frac{\lambda}{2} \sum_{i=1}^{n_m} ||\mathbf{v}^{(i)}||^2$$

In [10]:
# ── 2. Collaborative Filtering ────────────────────────────────────────
# Use similarity between users/items to predict missing ratings

print("\n── Collaborative Filtering (Matrix Factorisation) ──")

# Ratings matrix (4 users × 5 movies), 0 = unknown
R = np.array([
    [5, 4, 0, 0, 1],
    [0, 0, 4, 5, 0],
    [3, 0, 0, 2, 4],
    [0, 3, 5, 0, 0],
], dtype=float)

n_users, n_items = R.shape
n_factors = 2   # latent features

# Initialise latent factors
rng = np.random.default_rng(42)
U = rng.normal(0, 0.1, (n_users, n_factors))   # user factors
V = rng.normal(0, 0.1, (n_items, n_factors))   # item factors


def train_collab(R, U, V, alpha=0.02, reg=0.1, n_iters=2000):
    """Alternating least squares on observed ratings only."""
    observed = R > 0
    for epoch in range(n_iters):
        # Predict
        pred = U @ V.T
        error = (observed * (R - pred))

        # Gradients
        dU = -(error @ V) + reg * U
        dV = -(error.T @ U) + reg * V

        U -= alpha * dU
        V -= alpha * dV

        loss = 0.5 * np.sum(observed * (R - U @ V.T) ** 2) + 0.5 * reg * (np.sum(U ** 2) + np.sum(V ** 2))
        if epoch % 500 == 0:
            print(f"  Iter {epoch:4d}  loss={loss:.4f}")

    return U, V


── Collaborative Filtering (Matrix Factorisation) ──


In [11]:
U, V = train_collab(R, U, V)
predicted_ratings = U @ V.T

print("\nOriginal ratings (0 = unknown):")
print(R)
print("\nPredicted ratings (filled):")
print(predicted_ratings.round(2))

  Iter    0  loss=72.8299
  Iter  500  loss=2.1044
  Iter 1000  loss=2.0658
  Iter 1500  loss=2.0557

Original ratings (0 = unknown):
[[5. 4. 0. 0. 1.]
 [0. 0. 4. 5. 0.]
 [3. 0. 0. 2. 4.]
 [0. 3. 5. 0. 0.]]

Predicted ratings (filled):
[[ 4.93  3.94  4.12  5.34  1.02]
 [ 4.43  3.63  4.    4.9   0.64]
 [ 2.97  1.34 -1.01  2.    3.9 ]
 [ 2.79  2.99  4.9   3.93 -1.84]]


## 3. Similarity-Based Recommendation

We can find items similar to a given target item by calculating the **Cosine Similarity** between their feature representation vectors:
$$\text{similarity}(\mathbf{u}, \mathbf{v}) = \frac{\mathbf{u} \cdot \mathbf{v}}{||\mathbf{u}|| \, ||\mathbf{v}||} = \frac{\sum_k u_k v_k}{\sqrt{\sum_k u_k^2} \sqrt{\sum_k v_k^2}}$$

This metric measures the cosine of the angle between two vectors, rendering it invariant to the vector magnitudes (which is useful, for example, if one movie has higher average rating values overall than another).

In [12]:
# ── 3. Similarity-Based Recommendation ────────────────────────────────

print("\n── Finding Similar Items (Cosine Similarity) ──")

sim_matrix = cosine_similarity(items)
print(f"Similarity matrix:\n{sim_matrix.round(3)}")

# For Movie A (index 0), find most similar
movie_idx = 0
similarities = sim_matrix[movie_idx]
most_similar = np.argsort(similarities)[-3:][::-1]
print(f"\nMost similar to {movie_names[movie_idx]}:")
for idx in most_similar:
    if idx != movie_idx:
        print(f"  {movie_names[idx]}  (similarity: {similarities[idx]:.3f})")


── Finding Similar Items (Cosine Similarity) ──
Similarity matrix:
[[1.    0.22  0.781 0.969 0.379]
 [0.22  1.    0.781 0.452 0.986]
 [0.781 0.781 1.    0.91  0.874]
 [0.969 0.452 0.91  1.    0.595]
 [0.379 0.986 0.874 0.595 1.   ]]

Most similar to Movie A:
  Movie D  (similarity: 0.969)
  Movie C  (similarity: 0.781)


## 4. Deep Learning Collaborative Filtering

In the neural network approach to collaborative/content-based filtering, user features and item features are passed through two separate neural networks (a **Siamese/Dual Network** architecture) to generate embedding vectors of equal size. The dot product of these user and item embeddings predicts the final user rating.

In [13]:
# ── 4. Deep Learning Recommender (Dual/Siamese Network) ─────────────────
# Predict rating using the dot product of user & item embedding networks

print("\n── Deep Learning Recommender ──")
import tensorflow as tf

# Set random seed for reproducibility
tf.random.set_seed(42)

# Synthetic features:
# Users: [age_scaled, romance_preference, action_preference] (4 users)
user_features = np.array([
    [0.1, 0.9, 0.1],  # User 0: young, likes romance
    [0.8, 0.1, 0.9],  # User 1: older, likes action
    [0.5, 0.5, 0.5],  # User 2: mid-age, balanced
    [0.2, 0.8, 0.2]   # User 3: young, likes romance
], dtype=np.float32)

# Movies: [year_scaled, romance_genre, action_genre] (5 movies)
movie_features = np.array([
    [0.9, 0.9, 0.1],  # Movie A: modern romance
    [0.1, 0.1, 0.9],  # Movie B: classic action
    [0.5, 0.5, 0.5],  # Movie C: balanced
    [0.8, 0.8, 0.2],  # Movie D: modern romance
    [0.2, 0.2, 0.8]   # Movie E: classic action
], dtype=np.float32)

# Generate a synthetic training dataset: user index, movie index, rating
# We train the networks to predict these observed ratings:
user_indices, movie_indices = np.where(R > 0)
y_train = R[R > 0].astype(np.float32)

# Extract features for training pairs
X_user_train = user_features[user_indices]
X_movie_train = movie_features[movie_indices]

# Dual network architecture (functional API)
user_input = tf.keras.layers.Input(shape=(3,), name="user_input")
user_net = tf.keras.layers.Dense(16, activation="relu")(user_input)
user_vec = tf.keras.layers.Dense(8, activation="linear", name="user_embedding")(user_net)

movie_input = tf.keras.layers.Input(shape=(3,), name="movie_input")
movie_net = tf.keras.layers.Dense(16, activation="relu")(movie_input)
movie_vec = tf.keras.layers.Dense(8, activation="linear", name="movie_embedding")(movie_net)

# Dot product of embeddings to predict rating
pred_rating = tf.keras.layers.Dot(axes=1, name="dot_product")([user_vec, movie_vec])

dl_model = tf.keras.Model(inputs=[user_input, movie_input], outputs=pred_rating)
dl_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.05), loss="mean_squared_error")

# Train
dl_model.fit([X_user_train, X_movie_train], y_train, epochs=200, verbose=0)

# Evaluate / Predict ratings for all user-movie pairs
all_users = np.repeat(np.arange(n_users), n_items)
all_movies = np.tile(np.arange(n_items), n_users)

X_user_all = user_features[all_users]
X_movie_all = movie_features[all_movies]

dl_preds = dl_model.predict([X_user_all, X_movie_all], verbose=0).reshape(n_users, n_items)
print("Predicted ratings using Deep Learning:")
print(dl_preds.round(2))


── Deep Learning Recommender ──


AttributeError: module 'numpy' has no attribute 'dtypes'

## Key Takeaways

- **Content-Based Filtering**: Recommends items by matching user preferences to item features. Can make predictions for **new items** as long as item features are available.
- **Collaborative Filtering**: Recommends items based on the similarity profiles of users. Learns latent representations and uncovers hidden affinities without requiring explicit feature labels.
- **Matrix Factorization**: Decomposes the ratings matrix as $R \approx U V^T$ to predict unobserved entries.
- **Cosine Similarity**: Measures vector direction closeness on a scale of $[-1, 1]$.
- **Cold Start Problem**: Occurs when a new user or item has zero history. Solution: Use content-based filtering until enough user interaction data is gathered.
- **Mean Normalization**: Subtracts the average rating of each user before matrix factorization. This accounts for users who are systematically generous or harsh critics, allowing the system to predict reasonable baseline ratings for users with minimal ratings.
- **Deep Learning Recommender**: Siamese/Dual Neural Networks project user and item feature vectors into a shared embedding space, predicting preferences via dot product.

In [ ]:
# ── Key Takeaways ─────────────────────────────────────────────────────
print("""
╔══ Key Takeaways ───────────────────────────────────────────────╗
║ • Content-based: predict rating from item features + user      ║
║   preferences → generalises to NEW items                       ║
║ • Collaborative: find latent factors from user-item matrix     ║
║   → generalises to NEW users (but cold-start problem)         ║
║ • Matrix Factorisation: R ≈ U·Vᵀ (user × item factors)        ║
║ • Cosine Similarity: measures item-item closeness             ║
║ • Cold Start: new user/item has no history → use content-based ║
║ • Mean Normalisation: subtract user's mean rating first        ║
║   → helps predict for users who rate everything high/low      ║
║ • Deep Learning: dual networks output user & item embeddings   ║
║   → predicts ratings via the dot product of learned vectors    ║
╚════════════════════════════════════════════════════════════════╝
""")